# 01 — Data Pull
Pull all raw data from IMF, World Bank, OWID, and EU ETS.
Run this notebook first. Outputs saved to `data/raw/`.

In [ ]:
import sys
sys.path.append('../src')
from data_pull import *
import pandas as pd
import os

RAW = '../data/raw/'
os.makedirs(RAW, exist_ok=True)

## 1. World Bank — Energy, GDP, Trade

In [ ]:
wb_indicators = {
    'NY.GDP.MKTP.CD': 'gdp_usd',
    'NY.GDP.MKTP.KD.ZG': 'gdp_growth',
    'NE.TRD.GNFS.ZS': 'trade_openness',
    'EG.IMP.CONS.ZS': 'energy_imports_pct',
    'EG.USE.PCAP.KG.OE': 'energy_use_per_capita',
    'EG.ELC.NUCL.ZS': 'nuclear_pct_electricity',
    'FP.CPI.TOTL.ZG': 'inflation_cpi',
    'EG.USE.COMM.FO.ZS': 'fossil_fuel_pct_energy',
    'TX.VAL.FUEL.ZS.UN': 'fuel_exports_pct_merchandise',
    'TM.VAL.FUEL.ZS.UN': 'fuel_imports_pct_merchandise',
}

wb_df = pull_world_bank(wb_indicators, start_year=1960, end_year=2023)
validate_pull(wb_df, 'World Bank WDI', min_rows=1000,
              required_cols=['gdp_usd', 'trade_openness'])
wb_df.head()

## 2. IMF COFER — Reserve Currency Composition

In [ ]:
cofer_df = pull_imf_cofer()
validate_pull(cofer_df, 'IMF COFER', min_rows=50)
if cofer_df is not None:
    print(cofer_df.shape)
    cofer_df.head()

## 3. OWID Energy Data (Energy Institute Statistical Review)
Our World in Data energy dataset — incorporates the Energy Institute Statistical Review
(formerly BP Statistical Review). Clean panel format, 1900–present, ~200 countries.
Downloaded directly from GitHub — no manual steps required.

In [ ]:
owid_df = pull_owid_energy()
validate_pull(owid_df, 'OWID Energy', min_rows=5000,
              required_cols=['country', 'year', 'primary_energy_consumption'])
if owid_df is not None:
    print(f"Countries: {owid_df['country'].nunique()}")
    print(f"Years: {owid_df['year'].min()}–{owid_df['year'].max()}")
    # Confirm our four focal countries are present
    focal = ['China', 'India', 'Russia', 'Japan']
    found = [c for c in focal if c in owid_df['country'].values]
    print(f"Focal countries found: {found}")
    owid_df.head()

## 4. EU ETS Compliance Data

In [ ]:
ets_df = pull_eu_ets()
validate_pull(ets_df, 'EU ETS Compliance', min_rows=100)
if ets_df is not None:
    print(ets_df.shape)
    ets_df.head()

## 5. Additional Sources

In [ ]:
# Chinn-Ito capital account openness
ci_df = pull_chinn_ito()
validate_pull(ci_df, 'Chinn-Ito KAOPEN', min_rows=500, required_cols=['kaopen'])

# World governance indicators
gov_df = pull_governance_indicators()
validate_pull(gov_df, 'WGI Governance', min_rows=200)

# Thorium reserves (hardcoded USGS 2024)
th_df = pull_thorium_reserves()
validate_pull(th_df, 'Thorium Reserves', min_rows=10)

# BIS FX turnover (hardcoded BIS Triennial 2022)
bis_df = pull_bis_fx_turnover()
validate_pull(bis_df, 'BIS FX Turnover', min_rows=40)

# OWID CO2
co2_df = pull_owid_co2()
validate_pull(co2_df, 'OWID CO2', min_rows=5000)

## 6. Summary — What We Have

In [ ]:
files = os.listdir(RAW)
print('Files in data/raw/:')
for f in sorted(files):
    size = os.path.getsize(os.path.join(RAW, f))
    print(f'  {f} ({size/1024:.1f} KB)')

print()
print('Manual downloads still required (see DATA_AUDIT.md):')
print('  1. EU ETS carbon prices  → sandbag.be/carbon-price-viewer')
print('  2. SWIFT RMB tracker     → swift.com RMB tracker PDFs')
print('  3. Russia fossil tracker → bruegel.org/dataset/russian-fossil-fuel-tracker')